Dưới đây là phiên bản nâng cao của notebook Google Colab:

  * **Trực quan hóa dữ liệu (Visualization):** Khám phá sâu hơn về dữ liệu.
  * **Giải thích mô hình (Explainable AI):** Sử dụng thư viện `SHAP` để hiểu tại sao mô hình đưa ra dự đoán như vậy.
  * **Lưu trữ (Save):** Lưu lại mô hình và các thành phần cần thiết để có thể triển khai trên một ứng dụng web (ví dụ: Streamlit).

Hãy thực hiện theo các bước tương tự như trước (Mở Colab, lấy Kaggle API) và chạy các ô code sau.

-----

### **Notebook: Pipeline Hoàn Chỉnh từ Phân tích đến Triển khai Mô hình**

#### **Bước 1: Cài đặt & Cấu hình**

Cài đặt tất cả các thư viện cần thiết, bao gồm `shap` để giải thích mô hình.

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q catboost kaggle shap

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import shap

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, classification_report
from catboost import CatBoostClassifier

# --- Cấu hình Kaggle API ---
# (Dán code cấu hình Kaggle API của bạn vào đây giống như lần trước)
# Tạo thư mục .kaggle
!mkdir -p ~/.kaggle

# Dán nội dung file kaggle.json của bạn vào đây
kaggle_credentials = {
    "username": "YOUR_USERNAME",  # <-- THAY BẰNG USERNAME KAGGLE CỦA BẠN
    "key": "YOUR_API_KEY"        # <-- THAY BẰNG API KEY KAGGLE CỦA BẠN
}

with open(os.path.join(os.path.expanduser('~'), '.kaggle', 'kaggle.json'), 'w') as f:
    json.dump(kaggle_credentials, f)

!chmod 600 ~/.kaggle/kaggle.json

# --- Tải Dataset ---
!kaggle datasets download -d fedesoriano/heart-failure-prediction -f heart.csv
print("✅ Tải dữ liệu thành công!")

-----

#### **Bước 2: Tải và Lưu trữ Dữ liệu Gốc**

Tải dữ liệu vào DataFrame và lưu một bản sao sạch để sử dụng sau này.

In [ ]:
# Tải dữ liệu vào DataFrame
df = pd.read_csv('heart.csv')

# Lưu một bản sao của dataset gốc để sử dụng trong app triển khai
df.to_csv('heart_disease_dataset_original.csv', index=False)

print("📊 Dữ liệu gốc (5 dòng đầu):")
print(df.head())
print(f"\nKích thước dữ liệu: {df.shape}")

-----

#### **Bước 3: Trực quan hóa & Phân tích Dữ liệu Khám phá (EDA)** 📈

Hiểu rõ hơn về dữ liệu qua các biểu đồ.

In [ ]:
print("--- Phân tích các thuộc tính số ---")
numerical_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
df[numerical_features].describe()

# Vẽ biểu đồ phân phối cho các thuộc tính số
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_features):
    plt.subplot(2, 3, i + 1)
    sns.histplot(df, x=col, hue='HeartDisease', kde=True)
    plt.title(f'Phân phối của {col}')
plt.tight_layout()
plt.show()

print("\n--- Phân tích các thuộc tính phân loại ---")
categorical_features = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# Vẽ biểu đồ cột cho các thuộc tính phân loại
plt.figure(figsize=(16, 12))
for i, col in enumerate(categorical_features):
    plt.subplot(3, 2, i + 1)
    sns.countplot(data=df, x=col, hue='HeartDisease', palette='viridis')
    plt.title(f'Phân phối của {col} theo Bệnh tim')
plt.tight_layout()
plt.show()

print("\n--- Phân tích tương quan ---")
# Chuyển đổi các cột object sang số để tính tương quan
df_corr = df.copy()
for col in df_corr.select_dtypes(include='object').columns:
    df_corr[col], _ = pd.factorize(df_corr[col])

# Vẽ heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Heatmap Tương quan giữa các Thuộc tính')
plt.show()

[Image of a data visualization dashboard]

-----

#### **Bước 4: Tiền xử lý Dữ liệu**

Tách dữ liệu và xây dựng pipeline tiền xử lý, giống như trước đây.

In [ ]:
# Tách biến mục tiêu (y) và các biến độc lập (X)
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# Chia dữ liệu thành tập train và test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Tạo pipeline tiền xử lý
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Giữ lại các cột không được xử lý nếu có
)

print("✅ Pipeline tiền xử lý đã sẵn sàng.")

-----

#### **Bước 5: Huấn luyện và Đánh giá Mô hình** 🚀

Xây dựng pipeline hoàn chỉnh và huấn luyện mô hình cuối cùng trên toàn bộ tập huấn luyện.

In [ ]:
# Tạo pipeline hoàn chỉnh với mô hình CatBoost
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', CatBoostClassifier(verbose=0, random_state=42, auto_class_weights='Balanced'))
])

# Huấn luyện pipeline trên tập train
final_pipeline.fit(X_train, y_train)

# Đánh giá mô hình trên tập test
y_pred = final_pipeline.predict(X_test)
y_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]

print("--- Báo cáo Đánh giá trên Tập Test ---")
print(classification_report(y_test, y_pred))
print(f"AUC-ROC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("✅ Huấn luyện và đánh giá mô hình thành công.")

-----

#### **Bước 6: Giải thích Mô hình với SHAP (Explainable AI)** 🧠

Đây là phần quan trọng giúp hiểu "hộp đen" AI. Chúng ta sẽ xem những yếu tố nào ảnh hưởng nhiều nhất đến dự đoán.

In [ ]:
# Áp dụng preprocessor cho tập train để có dữ liệu đã được biến đổi
X_train_processed = final_pipeline.named_steps['preprocessor'].fit_transform(X_train)
X_test_processed = final_pipeline.named_steps['preprocessor'].transform(X_test)

# Lấy tên các feature sau khi đã OneHotEncode
feature_names = final_pipeline.named_steps['preprocessor'].get_feature_names_out()

# Tạo SHAP explainer
explainer = shap.TreeExplainer(final_pipeline.named_steps['classifier'])
shap_values = explainer.shap_values(X_test_processed)

# Trực quan hóa kết quả SHAP
print("\n--- Giải thích Mô hình với SHAP ---")

# Biểu đồ tổng quan các feature quan trọng nhất
print("Biểu đồ tổng quan mức độ quan trọng của các thuộc tính:")
shap.summary_plot(shap_values, X_test_processed, feature_names=feature_names, plot_type="bar")

# Biểu đồ chi tiết hơn về tác động của từng feature
print("\nBiểu đồ chi tiết về tác động của các thuộc tính (Beeswarm):")
shap.summary_plot(shap_values, X_test_processed, feature_names=feature_names)

-----

#### **Bước 7: Lưu Model và các Thành phần để Triển khai** 💾

Lưu pipeline hoàn chỉnh (bao gồm cả preprocessor và model) vào một file duy nhất. Đây là cách tốt nhất để đảm bảo tính nhất quán khi triển khai.

In [ ]:
# Tên file để lưu
pipeline_filename = 'catboost_heart_disease_pipeline.pkl'

# Sử dụng pickle để lưu pipeline
with open(pipeline_filename, 'wb') as f:
    pickle.dump(final_pipeline, f)

print(f"✅ Pipeline hoàn chỉnh đã được lưu vào file: '{pipeline_filename}'")
print("Tệp này chứa cả preprocessor và model, sẵn sàng để tải và sử dụng trong ứng dụng Streamlit.")

-----

#### **Bước 8: (Tùy chọn) Tải lại và Kiểm tra Mô hình**

Bước này mô phỏng cách ứng dụng Streamlit của bạn sẽ tải mô hình và sử dụng nó để dự đoán.

In [ ]:
# Tải lại pipeline từ file
with open(pipeline_filename, 'rb') as f:
    loaded_pipeline = pickle.load(f)

# Lấy một mẫu dữ liệu từ tập test để kiểm tra
sample = X_test.iloc[[0]]
print("--- Thử nghiệm dự đoán trên một mẫu dữ liệu ---")
print("Dữ liệu đầu vào:")
print(sample)

# Dự đoán
prediction = loaded_pipeline.predict(sample)[0]
prediction_proba = loaded_pipeline.predict_proba(sample)[0][1]

print(f"\n=> Dự đoán của mô hình: {'Có bệnh tim' if prediction == 1 else 'Không có bệnh tim'}")
print(f"=> Xác suất có bệnh tim: {prediction_proba:.2%}")

print("\n✅ Quá trình tải lại và dự đoán thử nghiệm thành công!")

Giờ đây bạn đã có một notebook hoàn chỉnh, không chỉ xây dựng mô hình mà còn hiểu, giải thích và đóng gói nó để sẵn sàng cho bước tiếp theo: xây dựng ứng dụng demo.